# Lesson 3: The loop

**Project:** Agentic Engineering Worklog
**Model:** `qwen3:8b`, running on this PC through Ollama

This is the lesson where the work log program becomes an **agent**. The change in code is small, a loop around the model call, but it moves a big decision from your code to the model: *what should happen next?*

## Where lesson 2 left off

Lesson 2 gave the model one tool, `add_entry`, and scored 6 of 6. Its program, `run_one_turn`, had fixed steps:

1. Ask the model.
2. If it requests tools, run them.
3. Ask the model once more for a final answer.

The model got **one** chance to ask for tools. That's fine for "log this", but some requests need more than one action:

> Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.

That needs a save *and* a search, and the search only gives the right total if the save has already happened.

## The big idea: what is a loop, and why does it make an agent?

A **loop** is code that repeats. Here it repeats this cycle:

1. Send the conversation to the model.
2. If the model asks for tools, run them, add the results to the conversation, and **go back to 1**.
3. If the model answers without asking for anything, stop: that's the final answer.

The key difference from lesson 2: **the model decides when it's finished.** It can look at a tool's result and then choose another tool, as many times as it needs. Your code no longer fixes the number of steps.

That's the definition of an agent from the start of this project: a model working toward a goal in a loop, choosing its own next step from what it has seen so far.

Your code still sets the limits:

- **which tools exist:** the model can only ask for those.
- **how many rounds it gets:** a step limit, so it can't loop forever.

## What we are looking for

By the end of this notebook you should be able to say yes to five things:

1. I have a second tool, `search_entries`, that reads the log back.
2. I have seen how lesson 2's one-round program handles a two-step request.
3. I can write the loop: call the model, run what it asks for, send the results back, and repeat until it stops asking.
4. I know why the loop needs a step limit.
5. I can score the agent on a test set that includes multi-step requests.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter**. Run the cells in order from top to bottom.

## Step 1: Setup

### What we are looking for

The same setup as lesson 2, plus **today's date**, which the model needs to understand words like "today" and "yesterday".

In [1]:
import json
import re
from datetime import date, datetime, timedelta
from pathlib import Path

import ollama

MODEL = "qwen3:8b"

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

DATA_DIR = project_root / "data"
DATA_DIR.mkdir(exist_ok=True)

WORK_LOG = DATA_DIR / "work_log.jsonl"
LOG_FILE = WORK_LOG  # the file the tools read and write; Step 8 points it at a test file for a while

TODAY = date.today()

print("Log file:", LOG_FILE)
print("Today:   ", TODAY.isoformat())

Log file: B:\_Dev\Python\Agentic-Engineering-Worklog\data\work_log.jsonl
Today:    2026-11-16


### The code, line by line

- The imports are lesson 2's, plus:
    - `re` (**regular expressions**): a tool for finding patterns in text, used in Step 8 to check answers.
    - `date` and `timedelta`: for today's date, and for working out yesterday's.
- `DATA_DIR = project_root / "data"` is the data folder, and `mkdir(exist_ok=True)` creates it if it's missing.
- `WORK_LOG` is the path to your real work log.
- `LOG_FILE = WORK_LOG` is the file the tools actually use. It's a separate name on purpose: Step 8 points `LOG_FILE` at a test file for a while, then back again.
- `TODAY = date.today()` gets today's date from the PC's clock.

### How to read it

Your run printed:

```
Log file: B:\_Dev\Python\Agentic-Engineering-Worklog\data\work_log.jsonl
Today:    2026-11-16
```

*Since this run, the projects moved from `B:\_Dev\Python` to `B:\Development\Python`. The saved output above still shows the old path, because it's a record of what ran. Running the cell again would show the new one.*

### What it means

This project's log started empty. The `data` folder is listed in `.gitignore` (a file that tells Git what *not* to upload), so your real work log never goes to GitHub.

**Why the model needs the date:** a model has no clock. Unless today's date is in its messages, it can't know what "today" means, and it would have to guess a date. Step 3 puts the date into its instructions.

### What's next and why

The agent needs a way to read the log back. Step 2 adds that tool.

## Step 2: Two tools as plain Python

### What we are looking for

The new tool, `search_entries`, working by hand before the model touches it. `add_entry` is unchanged from lesson 2.

In [2]:
def read_entries():
    """Every entry in the current log file, oldest first."""
    if not LOG_FILE.exists():
        return []
    lines = LOG_FILE.read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]


def add_entry(project, description, hours=None):
    """Save one work log entry and return a short confirmation."""
    entry = {
        "logged_at": datetime.now().isoformat(timespec="seconds"),
        "project": project,
        "description": description,
        "hours": hours,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
    return f"Saved entry for {project}."


def search_entries(project=None, date_from=None, date_to=None):
    """Find entries by project and date range (YYYY-MM-DD, both ends included). Returns text with a total."""
    matches = []
    for entry in read_entries():
        day = entry["logged_at"][:10]
        if project and entry["project"].lower() != project.lower():
            continue
        if date_from and day < date_from:
            continue
        if date_to and day > date_to:
            continue
        matches.append(entry)

    if not matches:
        return "No matching entries."

    lines = []
    total = 0.0
    for entry in matches:
        hours = entry["hours"]
        if hours is not None:
            total += float(hours)
        shown = f"{hours} h" if hours is not None else "hours not given"
        lines.append(f"{entry['logged_at'][:10]} | {entry['project']} | {entry['description']} | {shown}")
    lines.append(f"{len(matches)} entries. Total hours where given: {total:g}")
    return "\n".join(lines)


print(add_entry("Agentic Engineering Worklog", "Tested both tools by hand, no model involved", 0.25))
print()
print(search_entries(project="Agentic Engineering Worklog", date_from=TODAY.isoformat()))

Saved entry for Agentic Engineering Worklog.

2026-11-16 | Agentic Engineering Worklog | Tested both tools by hand, no model involved | 0.25 h
1 entries. Total hours where given: 0.25


### The code, line by line

**`read_entries()`** reads the whole log:
- if the file doesn't exist yet, it returns an empty list.
- otherwise it reads every line and turns each one back from JSON text into a Python dictionary with `json.loads`. `if line.strip()` skips blank lines.

**`add_entry(...)`** is lesson 2's tool, unchanged.

**`search_entries(project=None, date_from=None, date_to=None)`** finds matching entries. All three inputs are optional:
- for each entry, `entry["logged_at"][:10]` takes the first 10 characters of the timestamp, which is just the date, such as `2026-11-16`.
- `continue` means "skip this entry and go to the next". An entry is skipped if:
    - a project was given and it doesn't match (`.lower()` makes the comparison ignore capital letters);
    - it's before `date_from`, or after `date_to`.
- Dates written as `YYYY-MM-DD` can be compared like text: `"2026-11-15" < "2026-11-16"` is `True`. That's why this date format is used.
- The rest builds the answer: one line per match, and a last line with the count and the **total hours**. `{total:g}` prints the number without needless zeros: `2.5`, not `2.500000`.

The last lines save one test entry and search for it.

### How to read it

Your run printed:

```
Saved entry for Agentic Engineering Worklog.

2026-11-16 | Agentic Engineering Worklog | Tested both tools by hand, no model involved | 0.25 h
1 entries. Total hours where given: 0.25
```

The save worked, and the search found it, with a total of 0.25 hours.

### What it means

Two design choices in `search_entries` matter later:

- **Code adds up the hours, not the model.** Small models make arithmetic slips. Handing the model a finished total means it only has to *repeat* a number, not *calculate* one. Wherever an exact answer is needed, let code do it.
- **It returns plain, labelled text.** The model reads this to answer you, and short lines like `date | project | description | hours` are easy for it to read correctly.

### What's next and why

The model can't see these functions. Step 3 describes both tools to it, and writes its instructions.

## Step 3: Describe both tools and write the instructions

### What we are looking for

The descriptions the model will read, and a system prompt with the rules it needs for two-step requests.

In [3]:
ADD_ENTRY_TOOL = {
    "type": "function",
    "function": {
        "name": "add_entry",
        "description": "Save one work log entry. Use this when the user describes work they did.",
        "parameters": {
            "type": "object",
            "properties": {
                "project": {"type": "string", "description": "Name of the project the work was for."},
                "description": {"type": "string", "description": "Short description of what was done."},
                "hours": {"type": "number", "description": "Hours spent. Leave this out if the user did not say."},
            },
            "required": ["project", "description"],
        },
    },
}

SEARCH_ENTRIES_TOOL = {
    "type": "function",
    "function": {
        "name": "search_entries",
        "description": (
            "Look up saved work log entries. Use this to answer questions about past work. "
            "Returns the matching entries and the total hours."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "project": {"type": "string", "description": "Only entries for this project. Leave out for all projects."},
                "date_from": {"type": "string", "description": "First day to include, as YYYY-MM-DD."},
                "date_to": {"type": "string", "description": "Last day to include, as YYYY-MM-DD."},
            },
            "required": [],
        },
    },
}

TOOLS = [ADD_ENTRY_TOOL, SEARCH_ENTRIES_TOOL]
AVAILABLE_TOOLS = {"add_entry": add_entry, "search_entries": search_entries}

KNOWN_PROJECTS = [
    "Cornerpost Watch", "Cornerpost Logic", "Herd Planner", "Agentic Engineering Worklog",
]

SYSTEM_PROMPT = (
    "You are a work log assistant. "
    f"Today is {TODAY.isoformat()} ({TODAY.strftime('%A')}). "
    "When the user describes work they did, call add_entry. "
    "When the user asks about past work, call search_entries and answer from its result. "
    "If a request needs both, save first and search after. "
    "Known projects: " + ", ".join(KNOWN_PROJECTS) + ". "
    "Use one of those names when the work matches one. "
    "Only include hours if the user stated them. Convert minutes to hours. "
    "If the message is not about work, answer normally without calling a tool."
)


def call_model(messages):
    """One call to the local model with both tools on offer."""
    return ollama.chat(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        think=False,
        options={"temperature": 0},
    )


print(SYSTEM_PROMPT)

You are a work log assistant. Today is 2026-11-16 (Monday). When the user describes work they did, call add_entry. When the user asks about past work, call search_entries and answer from its result. If a request needs both, save first and search after. Known projects: Cornerpost Watch, Cornerpost Logic, Herd Planner, Agentic Engineering Worklog. Use one of those names when the work matches one. Only include hours if the user stated them. Convert minutes to hours. If the message is not about work, answer normally without calling a tool.


### The code, line by line

- `ADD_ENTRY_TOOL` is the same as lesson 2.
- `SEARCH_ENTRIES_TOOL` describes the new tool: "Look up saved work log entries. Use this to answer questions about past work." Its three inputs are all optional (`"required": []`), and the date descriptions say the exact format, `YYYY-MM-DD`.
- `TOOLS` offers both tools to the model.
- `AVAILABLE_TOOLS` is the lookup table from lesson 2, now with two entries.
- `KNOWN_PROJECTS` now uses "Agentic Engineering Worklog" as the project name for this work.
- `SYSTEM_PROMPT` is built with an **f-string** (the `f` before the quotes), which fills `{...}` with values. Here it adds today's date and the day of the week (`%A` means the full day name).
- `call_model(messages)` is lesson 2's call, with both tools on offer.

### How to read it

Your run printed the finished system prompt, with today's date filled in:

> You are a work log assistant. Today is 2026-11-16 (Monday). When the user describes work they did, call add_entry. When the user asks about past work, call search_entries and answer from its result. If a request needs both, save first and search after. Known projects: ... Only include hours if the user stated them. Convert minutes to hours. ...

### What it means

Each rule in the prompt heads off a specific mistake:

| Rule | Mistake it prevents |
|---|---|
| Today is 2026-11-16 (Monday) | Guessing what "today" and "yesterday" mean |
| call search_entries and answer from its result | Answering questions about your work from imagination |
| save first and search after | Searching before the new entry exists, so the total is too low |
| Convert minutes to hours | Saving "30 minutes" as `30` hours |

Writing a system prompt is mostly this: noticing a likely mistake, and adding one clear sentence that prevents it.

### What's next and why

Before building the loop, Step 4 tries the two-step request with lesson 2's one-round program, to see what the loop has to improve on.

## Step 4: Try the two-step request without a loop

### What we are looking for

How lesson 2's `run_one_turn`, which allows only one round of tools, handles a request that needs a save *and* a search.

In [4]:
TWO_STEP_REQUEST = (
    "Log 1 hour on Herd Planner updating the calving chart, "
    "then tell me how many hours I've logged on Herd Planner today."
)


def run_one_turn(user_text):
    """Lesson 2's approach: one round of tool requests, then a final reply."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    response = call_model(messages)
    if not response.message.tool_calls:
        return response.message.content

    messages.append(response.message)
    for call in response.message.tool_calls:
        name = call.function.name
        arguments = dict(call.function.arguments)
        result = AVAILABLE_TOOLS[name](**arguments)
        print(f"  [tool] {name}({arguments})")
        messages.append({"role": "tool", "tool_name": name, "content": result})

    final = call_model(messages)
    if final.message.tool_calls:
        wanted = [c.function.name for c in final.message.tool_calls]
        print(f"  The model asked for more tools ({', '.join(wanted)}), but this program has no second round.")
    return final.message.content


print("You:  ", TWO_STEP_REQUEST)
reply = run_one_turn(TWO_STEP_REQUEST)
print("Model:", repr(reply))

You:   Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.
  [tool] add_entry({'description': 'Updating the calving chart', 'hours': 1, 'project': 'Herd Planner'})
  [tool] search_entries({'date_from': '2026-11-16', 'date_to': '2026-11-16', 'project': 'Herd Planner'})
Model: 'You have logged 1 hour on Herd Planner today.'


### The code, line by line

- `TWO_STEP_REQUEST` is the test sentence. The parentheses let it be written over two lines.
- `run_one_turn` is lesson 2's function: ask once, run any tools requested, ask once more.
- One addition at the end: if the model's *final* reply asks for **more** tools, this program can't run them, so it prints a note saying so.
- `repr(reply)` shows the reply with quote marks, so an empty reply is visible as `''`.

### How to read it

Your run printed:

```
[tool] add_entry({'description': 'Updating the calving chart', 'hours': 1, 'project': 'Herd Planner'})
[tool] search_entries({'date_from': '2026-11-16', 'date_to': '2026-11-16', 'project': 'Herd Planner'})
Model: 'You have logged 1 hour on Herd Planner today.'
```

The model asked for **both tools at once**, in a single round, with the save first and the search second. The program runs requests in the order they're listed, so the save happened before the search, and the answer was right: at that moment, there was 1 hour on Herd Planner today.

### What it means

This was a surprise. The step was expected to show the one-round program failing, and it didn't, because the model **planned both actions up front**. A model can request several tools in one reply. It worked here because:

1. nothing about the search depended on the *result* of the save, so the model could plan both in advance; and
2. it listed them in the right order, following "save first and search after".

There are other ways this could have gone. If the model had listed the search first, the total would have been missing the new hour. And if it had asked for the search only *after* seeing the save's result, this program would have had no second round to run it in.

The deeper point: the one-round program works only when the model can plan everything from the start. It breaks when a step **depends on what an earlier step returned**, such as "if I've logged less than 5 hours today, log one more". Exercise 4 at the end tries exactly that.

This cell really did save one entry to your log. That's fine: it's real work you'll see in later searches.

### What's next and why

Step 5 builds the loop, which gives the model as many rounds as it needs.

## Step 5: The loop

### What we are looking for

The agent itself: a function that keeps calling the model and running its tools until it gives a final answer, with a safety limit.

In [5]:
MAX_STEPS = 6


def run_agent(user_text, verbose=True):
    """Let the model call tools as many times as it needs, up to MAX_STEPS. Returns (reply, tool calls made)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    calls_made = []

    for step in range(1, MAX_STEPS + 1):
        response = call_model(messages)
        tool_calls = response.message.tool_calls or []

        if not tool_calls:
            if verbose:
                print(f"  step {step}: answered")
            return response.message.content, calls_made

        messages.append(response.message)
        for call in tool_calls:
            name = call.function.name
            arguments = dict(call.function.arguments)
            if name not in AVAILABLE_TOOLS:
                result = f"Error: there is no tool named {name}."
            else:
                try:
                    result = AVAILABLE_TOOLS[name](**arguments)
                except TypeError as error:
                    result = f"Error: {error}"
            calls_made.append((name, arguments))
            if verbose:
                first_line = result.splitlines()[0] if result else ""
                print(f"  step {step}: {name}({arguments}) -> {first_line}")
            messages.append({"role": "tool", "tool_name": name, "content": result})

    return f"Stopped after {MAX_STEPS} steps without a final answer.", calls_made


print("Loop ready. Step limit:", MAX_STEPS)

Loop ready. Step limit: 6


### The code, line by line

- `MAX_STEPS = 6` is the most rounds the agent may take.
- `run_agent(user_text, verbose=True)` starts a message list (system prompt plus your message) and an empty `calls_made` list to record every tool used. `verbose=True` means "print each step"; Step 8 turns that off to keep its output short.
- `for step in range(1, MAX_STEPS + 1):` repeats for step 1, 2, 3 and so on, up to 6. (`range` stops *before* its second number, hence the `+ 1`.)
- Inside the loop:
    1. Call the model.
    2. `tool_calls = response.message.tool_calls or []` gets the requested tools, or an empty list if there are none.
    3. **If there are none, the model is done:** return its answer. `return` ends the function, and with it the loop.
    4. Otherwise, add the model's request to the list, then run each tool:
        - if the name isn't in `AVAILABLE_TOOLS`, the result is an error message.
        - the function runs inside `try` / `except`, so a bad input becomes error text instead of a crash.
        - the result is added to the list as a `tool` message.
    5. Go round again. The model now sees everything that's happened so far.
- If all 6 rounds are used without a final answer, it returns a message saying it stopped.

### How to read it

Your run printed `Loop ready. Step limit: 6`. Nothing has been sent to the model yet; this cell only defines the function.

### What it means

Compare `run_agent` with `run_one_turn` from Step 4. The real change is the `for step in ...` loop around the model call. That small change hands the "what next?" decision to the model.

Two safety features matter in every real agent:

- **The step limit.** A confused model might keep searching again and again without settling on an answer. Without a limit, it would run forever. With one, it stops after 6 rounds and says so.
- **Errors go back as text.** A wrong tool name or a bad input doesn't crash the program. The model reads the error on the next round and can try again.

### What's next and why

Step 6 gives the agent the same two-step request.

## Step 6: Run the agent on the two-step request

### What we are looking for

The agent completing the request, with each round printed so you can see its decisions.

In [6]:
print("You:  ", TWO_STEP_REQUEST)
reply, calls = run_agent(TWO_STEP_REQUEST)
print("Model:", reply)

You:   Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.
  step 1: add_entry({'description': 'Updating the calving chart', 'hours': 1, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.
  step 1: search_entries({'date_to': '2026-11-16', 'project': 'Herd Planner', 'date_from': '2026-11-16'}) -> 2026-11-16 | Herd Planner | Updating the calving chart | 1 h
  step 2: answered
Model: You have logged 2 hours on Herd Planner today.


### The code, line by line

Three lines: print the request, run the agent, print its final reply. `reply, calls = ...` stores the two things `run_agent` returns: the answer and the list of tools used.

### How to read it

Your run printed:

```
step 1: add_entry({... 'hours': 1, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.
step 1: search_entries({... 'project': 'Herd Planner', 'date_from': '2026-11-16'}) -> 2026-11-16 | Herd Planner | Updating the calving chart | 1 h
step 2: answered
Model: You have logged 2 hours on Herd Planner today.
```

- **Step 1:** two tool calls in the same round, the save then the search, just as in Step 4.
- **Step 2:** a final answer.
- **2 hours** is right: 1 hour saved in Step 4, plus the 1 hour saved here.

### What it means

The agent got it right, and again it planned both actions at once.

So for *this* request, the loop and the one-round program behaved the same way. What the loop adds is the **option** of another round: if the model had needed to see the search result before deciding anything else, the loop would have given it that chance. The one-round program would not.

That's worth remembering: an agent isn't valuable because it always takes many steps. It's valuable because it *can* take more steps when the situation needs them, and it decides when.

### What's next and why

Step 7 checks that the same loop handles simpler requests too: questions that only need a search, and questions that need no tool.

## Step 7: Questions that only need a search

### What we are looking for

The right tool, or no tool, for three different kinds of question.

In [7]:
for text in [
    "How many hours have I logged on Herd Planner today?",
    "What have I worked on today?",
    "How many days are in a leap year?",
]:
    print("You:  ", text)
    reply, calls = run_agent(text)
    print("Model:", reply)
    print()

You:   How many hours have I logged on Herd Planner today?
  step 1: search_entries({'date_from': '2026-11-16', 'date_to': '2026-11-16', 'project': 'Herd Planner'}) -> 2026-11-16 | Herd Planner | Updating the calving chart | 1 h
  step 2: answered
Model: You have logged 2 hours on Herd Planner today.

You:   What have I worked on today?
  step 1: search_entries({'date_from': '2026-11-16', 'date_to': '2026-11-16'}) -> 2026-11-16 | Agentic Engineering Worklog | Tested both tools by hand, no model involved | 0.25 h
  step 2: answered
Model: Today, November 16, 2026, you worked on the following:

1. **Agentic Engineering Worklog**: Tested both tools by hand, no model involved (0.25 hours).
2. **Herd Planner**: Updating the calving chart (1 hour).
3. **Herd Planner**: Updating the calving chart (1 hour).

Total hours worked today: 2.25 hours.

You:   How many days are in a leap year?
  step 1: answered
Model: A leap year has 366 days.



### The code, line by line

A `for` loop sends three questions to `run_agent`, printing each question and its answer.

### How to read it

Your run printed (shortened):

| Question | What the agent did | Answer |
|---|---|---|
| How many hours have I logged on Herd Planner today? | one search | 2 hours |
| What have I worked on today? | one search, all projects | 3 entries listed, 2.25 hours in total |
| How many days are in a leap year? | no tool | 366 days |

Check the second answer against the log: 0.25 hours from the Step 2 test, plus two Herd Planner entries of 1 hour each (Steps 4 and 6), is 2.25 hours. Right.

### What it means

One program now handles three kinds of request without any special code for each: saving, looking up, and plain questions. On every round, the model chooses.

Notice it left out the project for "What have I worked on today?", so the search covered all projects. It chose its inputs to fit the question.

### What's next and why

These were good results on a handful of tries. Step 8 measures the agent properly, on a test set with known answers.

## Step 8: Measure it

### What we are looking for

Scores on six test requests, including two-step ones, each checked three ways:

- **Tools:** did it use the right tools, and no others?
- **Answer:** does the reply contain the right facts?
- **Saved:** were the right entries saved, with the right project and hours?

Each test starts from the same four entries, in a separate test file, so the right answers are known and your real log is never touched.

In [8]:
TEST_LOG = DATA_DIR / "test_log_lesson3.jsonl"
YESTERDAY = (TODAY - timedelta(days=1)).isoformat()
TODAY_S = TODAY.isoformat()

STARTING_ENTRIES = [
    {"logged_at": f"{YESTERDAY}T09:00:00", "project": "Cornerpost Watch", "description": "Tuned the cloud mask", "hours": 1},
    {"logged_at": f"{YESTERDAY}T14:00:00", "project": "Agentic Engineering Worklog", "description": "Ran lesson 2", "hours": None},
    {"logged_at": f"{TODAY_S}T08:30:00", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2},
    {"logged_at": f"{TODAY_S}T10:00:00", "project": "Herd Planner", "description": "Added vaccine records", "hours": 1.5},
]

TEST_CASES = [
    {"text": "Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.",
     "tools": {"add_entry", "search_entries"}, "answer": [r"\b2\.5\b"],
     "saved": [("Herd Planner", 1)]},
    {"text": "How many hours have I spent on Cornerpost Watch in total?",
     "tools": {"search_entries"}, "answer": [r"\b3(\.0)?\b"], "saved": []},
    {"text": "What did I work on yesterday?",
     "tools": {"search_entries"}, "answer": [r"cloud mask", r"lesson 2"], "saved": []},
    {"text": "Spent 30 minutes on the Cornerpost Logic company site.",
     "tools": {"add_entry"}, "answer": [], "saved": [("Cornerpost Logic", 0.5)]},
    {"text": "Log two things: 2 hours on Cornerpost Watch writing alert tests, and 1 hour on Herd Planner fixing the weight chart.",
     "tools": {"add_entry"}, "answer": [], "saved": [("Cornerpost Watch", 2), ("Herd Planner", 1)]},
    {"text": "What is the capital of Colorado?",
     "tools": set(), "answer": [r"denver"], "saved": []},
]


def reset_test_log():
    TEST_LOG.write_text("".join(json.dumps(e) + "\n" for e in STARTING_ENTRIES), encoding="utf-8")


def same_hours(got, expected):
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


scores = {"tools": 0, "answer": 0, "saved": 0}
LOG_FILE = TEST_LOG
try:
    for case in TEST_CASES:
        reset_test_log()
        reply, calls = run_agent(case["text"], verbose=False)
        new_entries = read_entries()[len(STARTING_ENTRIES):]

        tools_ok = {name for name, _ in calls} == case["tools"]
        answer_ok = all(re.search(p, reply or "", re.IGNORECASE) for p in case["answer"])
        got_saved = sorted((e["project"], e["hours"]) for e in new_entries)
        want_saved = sorted(case["saved"])
        saved_ok = len(got_saved) == len(want_saved) and all(
            g[0] == w[0] and same_hours(g[1], w[1]) for g, w in zip(got_saved, want_saved)
        )

        scores["tools"] += tools_ok
        scores["answer"] += answer_ok
        scores["saved"] += saved_ok

        print(case["text"])
        print(f"   tools: {'right' if tools_ok else 'WRONG'} {[name for name, _ in calls]}")
        print(f"   answer: {'right' if answer_ok else 'WRONG'}   saved: {'right' if saved_ok else 'WRONG'} {got_saved}")
        print(f"   reply: {(reply or '').strip()[:160]}")
        print()
finally:
    LOG_FILE = WORK_LOG

total = len(TEST_CASES)
print(f"Right tools: {scores['tools']} of {total}")
print(f"Right answer: {scores['answer']} of {total}")
print(f"Right saves: {scores['saved']} of {total}")
print()
print("Log file is back to:", LOG_FILE.name)

Log 1 hour on Herd Planner updating the calving chart, then tell me how many hours I've logged on Herd Planner today.
   tools: right ['add_entry', 'search_entries']
   answer: right   saved: right [('Herd Planner', 1)]
   reply: You have logged 2.5 hours on Herd Planner today.

How many hours have I spent on Cornerpost Watch in total?
   tools: right ['search_entries']
   answer: right   saved: right []
   reply: You have spent a total of 3 hours on Cornerpost Watch.

What did I work on yesterday?
   tools: right ['search_entries']
   answer: right   saved: right []
   reply: Yesterday, you worked on the following:

1. **Cornerpost Watch**: Tuned the cloud mask (1 hour)
2. **Agentic Engineering Worklog**: Ran lesson 2 (hours not spec

Spent 30 minutes on the Cornerpost Logic company site.
   tools: right ['add_entry']
   answer: right   saved: right [('Cornerpost Logic', 0.5)]
   reply: The entry for the Cornerpost Logic company site has been saved. Let me know if you need anything el

### The code, line by line

- `TEST_LOG` is a separate file just for testing.
- `STARTING_ENTRIES` are four known entries: two from yesterday, two from today. The dates are worked out from today, so the test still works on any day.
- `TEST_CASES` lists six requests. Each one says:
    - `tools`: which tools should be used (a **set**, written with `{}`, so order doesn't matter).
    - `answer`: patterns the reply must contain. `r"\b2\.5\b"` means "the number 2.5 as a whole number, not part of 12.5". These are **regular expressions**: `\b` marks the edge of a word, and `\.` means a real dot.
    - `saved`: the entries that should be added, as (project, hours).
- `reset_test_log()` rewrites the test file with just the four starting entries. It runs before every test, so tests can't affect each other.
- `LOG_FILE = TEST_LOG` points the tools at the test file. The `try` / `finally` block guarantees `LOG_FILE` is put back to your real log afterwards, **even if something goes wrong** mid-test.
- For each test, the code runs the agent quietly, reads any new entries, and checks tools, answer and saves. `re.IGNORECASE` means capital letters don't matter.

### How to read it

Your run printed one block per test, then:

```
Right tools: 6 of 6
Right answer: 6 of 6
Right saves: 6 of 6

Log file is back to: work_log.jsonl
```

The results, against the right answers worked out from the starting entries:

| Request | Right answer | What the agent did |
|---|---|---|
| Log 1 hour on Herd Planner, then tell me today's Herd Planner hours | 1.5 + 1 = **2.5** | saved, searched, said 2.5 |
| Hours on Cornerpost Watch in total | 1 + 2 = **3** | searched, said 3 |
| What did I work on yesterday? | the cloud mask, and lesson 2 | searched yesterday, listed both |
| Spent 30 minutes on the Cornerpost Logic company site | saved as **0.5** hours | saved 0.5 |
| Log two things: 2 hours on Cornerpost Watch, 1 hour on Herd Planner | **two** entries | two saves |
| What is the capital of Colorado? | Denver, no tool | Denver, no tool |

The last line confirms the tools point at your real log again.

### What it means

18 of 18 checks passed, and this test is harder than lesson 2's: it checks the *order* of steps, whether the agent searches instead of guessing, and whether one request with two items saves two entries.

A few things made this work, and they're worth carrying forward:

- **Code did the arithmetic.** The model repeated totals that `search_entries` had already added up.
- **The prompt rules worked:** "30 minutes" became 0.5, and "yesterday" became the right date.
- **The test set has a known starting state.** Without it, the right totals would change every time you logged real work.

Six tests is still a small set. A perfect score means "no problems found yet", not "no problems exist". The answer checks are simple pattern matches, too: if one ever shows WRONG, read the reply before deciding the agent got it wrong.

## Exercises

1. **Lower the step limit.** Set `MAX_STEPS = 1` in Step 5, and run Step 6 again. What does the agent do with a two-step request now? Set it back to 6 afterwards.
2. **Remove "save first and search after"** from the system prompt in Step 3, rerun Steps 3, 5 and 8, and compare the scores.
3. **Add a test case** for a project that isn't in the list, such as "Spent an hour on my portfolio site." What project name does the model choose?
4. **A request where the second step depends on the first.** Run this in a new cell:

    ```python
    reply, calls = run_agent(
        "How many hours have I logged on Herd Planner today? "
        "If it's less than 5, log 1 more hour of chart cleanup."
    )
    print(reply)
    ```

    The model can't know whether to log until it has seen the search result, so it should search in one round and decide in the next. Then try the same sentence with `run_one_turn` from Step 4, and compare.

## What it all means

- **An agent is a loop:** the model sees each tool's result and chooses the next step, until it decides it's done.
- Your code still decides *what the model can do* (the tools) and *how long it may try* (the step limit). The model decides *what to do next*.
- A model can plan several actions at once when they don't depend on each other. The loop matters when a step depends on an earlier result.
- Do exact work (sums, dates, saving) in code, and leave the language to the model.
- A test set with a known starting state turns "it seems to work" into scores you can compare after every change.

## Words used in this lesson

| Word | Meaning |
|---|---|
| Agent | A model working toward a goal in a loop, choosing its own next step from what it has seen. |
| Loop | Code that repeats. Here: call the model, run its tools, repeat until it answers. |
| Round / step | One pass through the loop: one model call, plus any tools it asked for. |
| Step limit | The most rounds the agent may take, so it can't run forever. |
| Search tool | A tool that reads saved data back, here `search_entries`. |
| f-string | Python text with `{...}` slots filled in by values, written `f"..."`. |
| Regular expression | A pattern for finding text, such as `\b2\.5\b`. |
| Set | A collection where order doesn't matter, written with `{}`. |
| `try` / `finally` | Code that always runs the `finally` part, even if something fails. |
| Known starting state | Test data that's the same every run, so the right answers are known. |

## What's next and why

**Lesson 4: Memory.** Every request to this agent starts fresh. If you say "Log an hour on Herd Planner" and then "Actually, make that two", the second request arrives on its own. Lesson 4 keeps the conversation between requests, adds a tool for correcting the last entry, and tests whether the agent can follow a back-and-forth the way a person would.